# AI-XRay · 03 — Entrenamiento + registro completo en MLflow

**Clase 4 · Laboratorio Práctico Dirigido — Proyecto Final Independiente**

### Objetivo de este notebook
1. Cargar los datasets de train/val construidos en `02_preprocessing.ipynb`.
2. Entrenar **3 variantes** del modelo (los runs de referencia del README) y registrar cada una como un run independiente y comparable en MLflow.
3. Ver, en cada run, el registro completo: parámetros, métricas por época, métricas finales (no solo accuracy), artefactos y el modelo con su *signature*.

> **Nota para el docente:** cada run puede tardar varios minutos por época (ResNet50 es una red grande). Si el tiempo de clase es limitado, ajustar `EPOCHS` a un valor bajo (ej. 3-5) para que el laboratorio en vivo alcance a completarse; el punto pedagógico es el *proceso* de comparación en MLflow, no que cada run llegue a converger del todo.

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import mlflow

from src.config import MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_URI, SPLIT_MANIFEST_PATH
from src.preprocessing import dataset_from_manifest
from src.train import compute_balanced_class_weight, train_and_log

split = pd.read_csv(SPLIT_MANIFEST_PATH)

train_ds = dataset_from_manifest(split, "train")
val_ds = dataset_from_manifest(split, "val")

class_weight = compute_balanced_class_weight(split[split["split"] == "train"]["label"])
print("class_weight:", class_weight)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
print("Tracking URI:", MLFLOW_TRACKING_URI)
print("Experiment:", MLFLOW_EXPERIMENT_NAME)


class_weight: {0: 1.0014285714285713, 1: 0.9985754985754985}
Tracking URI: sqlite:///C:\Users\juand\GitHub\M3-Cientifico-Datos-IA-Aplicada-DevSeniorCode\03_ia_aplicada_databricks_arquitecturas\clase_04_laboratorio_practico\AI-XRay\mlflow.db
Experiment: AI-XRay


In [5]:
# Ajusta este valor según el tiempo disponible en clase (ver nota del docente arriba).
EPOCHS = 3


## Run 1 — ResNet50 congelado, learning_rate=0.001, dropout=0.3

In [6]:
run_id_1 = train_and_log(
    train_ds, val_ds,
    run_name="resnet50_frozen_lr1e-3_do0.3",
    learning_rate=0.001, dropout=0.3, epochs=EPOCHS,
    fine_tune=False,
    class_weight=class_weight,
    tags={"fase": "A_congelado"},
)


Epoch 1/3


c:\Users\juand\anaconda3\envs\Clase-04-Py\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


44/44 - 68s - 2s/step - accuracy: 0.8067 - auc: 0.8927 - loss: 0.4105 - val_accuracy: 0.9103 - val_auc: 0.9744 - val_loss: 0.2177 - learning_rate: 0.0010
Epoch 2/3
44/44 - 54s - 1s/step - accuracy: 0.9101 - auc: 0.9638 - loss: 0.2450 - val_accuracy: 0.9169 - val_auc: 0.9791 - val_loss: 0.1904 - learning_rate: 0.0010
Epoch 3/3
44/44 - 50s - 1s/step - accuracy: 0.9080 - auc: 0.9676 - loss: 0.2299 - val_accuracy: 0.9136 - val_auc: 0.9817 - val_loss: 0.1875 - learning_rate: 0.0010
              precision    recall  f1-score   support

      NORMAL       0.94      0.89      0.91       150
   PNEUMONIA       0.89      0.94      0.92       151

    accuracy                           0.91       301
   macro avg       0.91      0.91      0.91       301
weighted avg       0.91      0.91      0.91       301

Run 'resnet50_frozen_lr1e-3_do0.3' completo -- run_id=6c5c909e114b4f17b7888d94f328bad0 -- métricas: {'accuracy': 0.9136212624584718, 'precision': 0.8930817610062893, 'recall': 0.9403973509933

## Run 2 — ResNet50 congelado, learning_rate=0.0001, dropout=0.5

In [7]:
run_id_2 = train_and_log(
    train_ds, val_ds,
    run_name="resnet50_frozen_lr1e-4_do0.5",
    learning_rate=0.0001, dropout=0.5, epochs=EPOCHS,
    fine_tune=False,
    class_weight=class_weight,
    tags={"fase": "A_congelado"},
)


Epoch 1/3


c:\Users\juand\anaconda3\envs\Clase-04-Py\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


44/44 - 75s - 2s/step - accuracy: 0.6484 - auc: 0.6890 - loss: 0.7002 - val_accuracy: 0.7741 - val_auc: 0.8803 - val_loss: 0.5099 - learning_rate: 1.0000e-04
Epoch 2/3
44/44 - 51s - 1s/step - accuracy: 0.7040 - auc: 0.7706 - loss: 0.5940 - val_accuracy: 0.8472 - val_auc: 0.9355 - val_loss: 0.4166 - learning_rate: 1.0000e-04
Epoch 3/3
44/44 - 49s - 1s/step - accuracy: 0.7553 - auc: 0.8331 - loss: 0.5044 - val_accuracy: 0.8804 - val_auc: 0.9532 - val_loss: 0.3584 - learning_rate: 1.0000e-04
              precision    recall  f1-score   support

      NORMAL       0.84      0.95      0.89       150
   PNEUMONIA       0.94      0.81      0.87       151

    accuracy                           0.88       301
   macro avg       0.89      0.88      0.88       301
weighted avg       0.89      0.88      0.88       301

Run 'resnet50_frozen_lr1e-4_do0.5' completo -- run_id=f1b2b3ecab5f467c962b431e7e7b3245 -- métricas: {'accuracy': 0.8803986710963455, 'precision': 0.9389312977099237, 'recall': 0.8

## Run 3 — ResNet50 + Fine-tuning, learning_rate=0.00001

Esta variante descongela las últimas capas de ResNet50 (a partir de la capa 143) y
entrena con un *learning rate* mucho más bajo -- ajustar todo el modelo con un LR alto
destruiría los pesos preentrenados de ImageNet en las primeras épocas.

In [8]:
run_id_3 = train_and_log(
    train_ds, val_ds,
    run_name="resnet50_finetuning_lr1e-5",
    learning_rate=0.00001, dropout=0.3, epochs=EPOCHS,
    fine_tune=True, fine_tune_at=143,
    class_weight=class_weight,
    tags={"fase": "B_fine_tuning"},
)


Epoch 1/3


c:\Users\juand\anaconda3\envs\Clase-04-Py\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


44/44 - 104s - 2s/step - accuracy: 0.7967 - auc: 0.9073 - loss: 0.4409 - val_accuracy: 0.8738 - val_auc: 0.9757 - val_loss: 0.2987 - learning_rate: 1.0000e-05
Epoch 2/3
44/44 - 81s - 2s/step - accuracy: 0.9108 - auc: 0.9722 - loss: 0.2426 - val_accuracy: 0.9203 - val_auc: 0.9804 - val_loss: 0.2302 - learning_rate: 1.0000e-05
Epoch 3/3
44/44 - 147s - 3s/step - accuracy: 0.9344 - auc: 0.9804 - loss: 0.1914 - val_accuracy: 0.9369 - val_auc: 0.9831 - val_loss: 0.2051 - learning_rate: 1.0000e-05
              precision    recall  f1-score   support

      NORMAL       0.90      0.99      0.94       150
   PNEUMONIA       0.99      0.89      0.93       151

    accuracy                           0.94       301
   macro avg       0.94      0.94      0.94       301
weighted avg       0.94      0.94      0.94       301

Run 'resnet50_finetuning_lr1e-5' completo -- run_id=bea5a2fd6c2249fea314dc6b41ef8357 -- métricas: {'accuracy': 0.9368770764119602, 'precision': 0.9852941176470589, 'recall': 0.8

In [9]:
run_ids = {"run_1": run_id_1, "run_2": run_id_2, "run_3": run_id_3}
print("Runs completados:", run_ids)


Runs completados: {'run_1': '6c5c909e114b4f17b7888d94f328bad0', 'run_2': 'f1b2b3ecab5f467c962b431e7e7b3245', 'run_3': 'bea5a2fd6c2249fea314dc6b41ef8357'}


## 🧪 Punto de decisión para tu Proyecto Final Independiente

Los 3 runs de arriba son el punto de partida de referencia (los mismos que dicta el
docente). Para tu propio proyecto, corre **al menos 2 variantes adicionales** cambiando
algo con criterio propio -- no al azar. Algunas ideas: otro valor de `dropout`, otra capa
de `fine_tune_at`, más/menos épocas, o (si usas el dataset completo) comparar con y sin
`class_weight`. Anota en una celda de markdown **por qué** probaste cada variante -- esa
justificación es, literalmente, el borrador de la sección 3 de tu Executive Summary.

### Cierre
Con los runs registrados en MLflow, continúa en `04_evaluation.ipynb` para compararlos y
registrar el mejor modelo.